# [Traces] H100 + Qwen3-30B-A3B + AIMO3: Transfer Validation

## Purpose
**Final validation** before tuning on gpt-oss-120b. Tests that strategies tuned on weaker models (Qwen3-4B/8B) **transfer to competition-level problems and models**.

## Why This Configuration?
- **H100 GPU**: Required for 30B model, available via AIMO competition
- **Qwen3-30B-A3B**: MoE model (3B active), strong math, fast inference
- **AIMO3 reference**: 53 actual competition problems with known answers

## Pipeline Position
```
1. Qwen3-4B + AIME       -> Tune strategies (11K traces)
2. Qwen3-8B + AIME-val   -> Validate on stronger model
3. Qwen3-30B + AIMO3     -> THIS: Competition-level check
4. gpt-oss-120b + AIMO3  -> Final tuning on exact model
```

## Output
53 problems x 12 samples = **636 traces**

In [ ]:
# Install vllm from offline wheels (no internet)
%pip install --no-index --find-links /kaggle/input/vllm-wheels-py312-cu129/wheels/ vllm

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os, sys, json, re, math, time, subprocess, tempfile
from collections import Counter
from typing import Optional, Dict, List, Any
import pandas as pd

os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

In [ ]:
class CFG:
    # Model - Qwen3-30B-A3B (MoE: 30B total, 3B active)
    model_name = 'Qwen/Qwen3-30B-A3B'
    model_path = '/kaggle/input/qwen-3/transformers/qwen3-30b-a3b/1'
    
    # Same prompts as competition notebook
    system_prompt_reasoning = (
        'You are a world-class International Mathematical Olympiad (IMO) competitor. '
        'The final answer must be a non-negative integer between 0 and 99999. '
        'You must place the final integer answer inside \\boxed{}.'
    )
    system_prompt_code = (
        'You are a world-class IMO competitor who excels at computational verification. '
        'Write Python code to explore and verify your reasoning whenever possible. '
        'The final answer must be a non-negative integer between 0 and 99999. '
        'You must place the final integer answer inside \\boxed{}.'
    )
    
    # 12 samples: 8 reasoning + 4 code-focused (match competition)
    prompt_configs = [('reasoning', system_prompt_reasoning)] * 8 + [('code', system_prompt_code)] * 4
    
    n_samples = 12
    max_turns = 16  # More turns for harder problems
    max_tokens = 8192
    temperature = 0.8
    top_p = 0.95
    
    # H100 settings
    gpu_memory_utilization = 0.92
    max_model_len = 16384
    
    problem_timeout = 360  # 6 min per problem
    code_timeout = 15
    
    output_dir = '/kaggle/working/traces'
    seed = 42

print(f"Config: {CFG.model_name}, {CFG.n_samples} samples")

In [ ]:
def load_aimo3_problems():
    """Load AIMO3 reference problems."""
    paths = [
        '/kaggle/input/aimo3-reference-problems/reference.csv',
        '/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv',
    ]
    for path in paths:
        if os.path.exists(path):
            df = pd.read_csv(path)
            print(f"Loaded {len(df)} AIMO3 problems from {path}")
            return df
    raise FileNotFoundError("AIMO3 reference not found")

df = load_aimo3_problems()
print(f"First problem: {df.iloc[0]['id']}")

In [ ]:
def execute_code(code: str, timeout: int = 15) -> str:
    full_code = "import math, numpy as np, sympy, itertools, functools\nfrom fractions import Fraction\n" + code
    try:
        with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
            f.write(full_code)
            f.flush()
            result = subprocess.run(['python', f.name], capture_output=True, text=True, timeout=timeout)
            os.unlink(f.name)
            if result.returncode != 0:
                return f"[ERROR] {result.stderr[:500]}"
            return result.stdout[:2000] or "[No output]"
    except subprocess.TimeoutExpired:
        return "[ERROR] Timeout"
    except Exception as e:
        return f"[ERROR] {str(e)[:200]}"

print(execute_code("import sympy; print(sympy.factorint(2024))"))

In [ ]:
from vllm import LLM, SamplingParams

model_path = CFG.model_path if os.path.exists(CFG.model_path) else CFG.model_name
print(f"Loading: {model_path}")

llm = LLM(
    model=model_path,
    gpu_memory_utilization=CFG.gpu_memory_utilization,
    max_model_len=CFG.max_model_len,
    trust_remote_code=True,
    seed=CFG.seed,
)
print("Model loaded")

In [ ]:
def extract_answer(text: str) -> Optional[int]:
    for pattern in [r'\\boxed\s*\{\s*([0-9,]+)\s*\}', r'answer\s*(?:is|=)\s*([0-9,]+)']:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            try:
                val = int(matches[-1].replace(',', ''))
                if 0 <= val <= 99999:  # AIMO range
                    return val
            except: pass
    return None

def extract_code_blocks(text: str) -> List[str]:
    return re.findall(r'```(?:python)?\s*\n(.*?)```', text, re.DOTALL | re.IGNORECASE)

def compute_entropy(logprobs: List[Dict]) -> float:
    if not logprobs: return float('inf')
    total, count = 0.0, 0
    for lp_dict in logprobs:
        if isinstance(lp_dict, dict) and lp_dict:
            ent = sum(-math.exp(lp) * math.log2(max(math.exp(lp), 1e-10)) for lp in lp_dict.values() if lp is not None)
            total += ent
            count += 1
    return total / count if count else float('inf')

In [ ]:
def solve_once(problem_text: str, system_prompt: str, seed: int) -> Dict[str, Any]:
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": problem_text}]
    tokenizer = llm.get_tokenizer()
    
    all_logprobs, code_executions = [], []
    answer, answer_source, last_code_output = None, None, None
    turns_used, total_tokens = 0, 0
    
    sampling_params = SamplingParams(temperature=CFG.temperature, top_p=CFG.top_p, max_tokens=CFG.max_tokens, seed=seed, logprobs=5)
    
    for turn in range(CFG.max_turns):
        turns_used = turn + 1
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        
        outputs = llm.generate([prompt], sampling_params)
        response = outputs[0].outputs[0]
        text = response.text
        total_tokens += len(response.token_ids)
        
        if response.logprobs:
            for lp in response.logprobs:
                if lp: all_logprobs.append({k: v.logprob for k, v in lp.items()})
        
        messages.append({"role": "assistant", "content": text})
        
        ans = extract_answer(text)
        if ans is not None:
            answer, answer_source = ans, "boxed"
            break
        
        code_blocks = extract_code_blocks(text)
        if code_blocks:
            outputs_list = []
            for code in code_blocks:
                output = execute_code(code, CFG.code_timeout)
                is_error = '[ERROR]' in output
                if not is_error: last_code_output = output
                outputs_list.append(output)
                code_executions.append({'turn': turn, 'code': code[:1000], 'output': output[:1000], 'is_error': is_error})
            messages.append({"role": "user", "content": f"Code output:\n```\n{''.join(outputs_list)}\n```\nContinue. Answer in \\boxed{{}}." })
        else:
            messages.append({"role": "user", "content": "Continue. Put answer in \\boxed{}."})
    
    if answer is None and last_code_output:
        for num in re.findall(r'\b(\d{1,5})\b', last_code_output):
            val = int(num)
            if 0 <= val <= 99999:
                answer, answer_source = val, "code_fallback"
                break
    
    entropy = compute_entropy(all_logprobs)
    if answer_source == "code_fallback": entropy = max(entropy, 8.0)
    
    return {'answer': answer, 'answer_source': answer_source, 'entropy': entropy, 'turns_used': turns_used,
            'total_tokens': total_tokens, 'code_executions': code_executions,
            'n_python_calls': len(code_executions), 'n_python_errors': sum(1 for c in code_executions if c['is_error'])}

In [ ]:
def generate_all_traces(df):
    os.makedirs(CFG.output_dir, exist_ok=True)
    
    config = {'model': CFG.model_name, 'n_samples': CFG.n_samples, 'max_turns': CFG.max_turns,
              'n_problems': len(df), 'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')}
    with open(f"{CFG.output_dir}/config.json", 'w') as f: json.dump(config, f, indent=2)
    
    all_results, total_correct, start_time = [], 0, time.time()
    
    for idx, row in df.iterrows():
        prob_id = str(row['id'])
        problem_text = row['problem']
        ground_truth = int(row['answer']) if pd.notna(row.get('answer')) else None
        
        print(f"\n{'='*60}")
        print(f"Problem {idx+1}/{len(df)} [{prob_id}] (GT={ground_truth})")
        
        attempts = []
        for i in range(CFG.n_samples):
            prompt_type, system_prompt = CFG.prompt_configs[i % len(CFG.prompt_configs)]
            seed = CFG.seed + idx * 100 + i * 7
            t0 = time.time()
            
            try:
                result = solve_once(problem_text, system_prompt, seed)
                result.update({'attempt_idx': i, 'seed': seed, 'wall_time_s': round(time.time()-t0, 2), 'prompt_type': prompt_type})
            except Exception as e:
                result = {'attempt_idx': i, 'answer': None, 'entropy': float('inf'), 'error': str(e)[:200], 'prompt_type': prompt_type}
            
            attempts.append(result)
            print(f"  {i+1}/{CFG.n_samples} [{prompt_type[:4]}]: ans={result.get('answer')}, ent={result.get('entropy', 0):.3f}")
        
        valid = [a['answer'] for a in attempts if a['answer'] is not None]
        default_answer = Counter(valid).most_common(1)[0][0] if valid else 0
        is_correct = default_answer == ground_truth if ground_truth is not None else None
        if is_correct: total_correct += 1
        
        print(f"  >> {'OK' if is_correct else 'X'} Default={default_answer}, Votes={dict(Counter(valid))}")
        
        trace = {'problem_id': prob_id, 'problem_text': problem_text, 'ground_truth': ground_truth,
                 'wall_time_s': round(time.time() - t0, 2), 'attempts': attempts,
                 'default_answer': default_answer, 'default_method': 'majority_vote', 'default_votes': dict(Counter(valid))}
        
        with open(f"{CFG.output_dir}/problem_{prob_id}.json", 'w') as f:
            json.dump(trace, f, indent=2, default=lambda x: str(x) if isinstance(x, float) and math.isinf(x) else x)
        
        all_results.append({'problem_id': prob_id, 'correct': is_correct, 'ground_truth': ground_truth})
        
        elapsed = time.time() - start_time
        print(f"  Progress: {idx+1}/{len(df)} | {total_correct} correct | {elapsed/60:.1f}min elapsed")
    
    total = sum(1 for r in all_results if r['correct'] is not None)
    accuracy = total_correct / total if total else 0
    
    summary = {'model': CFG.model_name, 'correct': total_correct, 'total': total, 'accuracy': round(accuracy, 4),
               'total_time_s': round(time.time() - start_time, 1), 'per_problem': all_results}
    with open(f"{CFG.output_dir}/summary.json", 'w') as f: json.dump(summary, f, indent=2)
    
    print(f"\n{'#'*60}")
    print(f"COMPLETE: {total_correct}/{total} ({accuracy*100:.1f}%)")
    print(f"Time: {(time.time()-start_time)/60:.1f} minutes")
    return summary

summary = generate_all_traces(df)

In [ ]:
print(f"\n{'='*60}")
print(f"AIMO3 Transfer Validation Results ({CFG.model_name})")
print(f"{'='*60}")
print(f"Accuracy: {summary['accuracy']*100:.1f}%")
print(f"Correct: {summary['correct']}/{summary['total']}")
print(f"Time: {summary['total_time_s']/60:.1f} minutes")
print(f"\nTraces saved to: {CFG.output_dir}")
print(f"\nNext step: Run gpt-oss-120b trace generator for final tuning")